# Fetch X Posts From KOLs

This notebook fetches recent posts from selected X/Twitter KOL accounts without using the official X API. It uses an authenticated `twikit` session with reusable cookies, stores a raw timeline archive, then creates a market/event-filtered dataset for downstream analysis.

In [1]:
import asyncio
import json
import os
import random
import re
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import pandas as pd
from twikit import Client


def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "Data" / "X").is_dir():
            return candidate
    return p


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Data" / "X"
RAW_DIR = PROJECT_ROOT / "Dataset" / "news" / "X" / "raw"
FILTERED_DIR = PROJECT_ROOT / "Dataset" / "news" / "X" / "filtered"
COOKIE_FILE = DATA_DIR / ".x_cookies.json"

for path in [RAW_DIR, FILTERED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

POSTS_PER_KOL = 200
MIN_SLEEP_SECONDS = 5
MAX_SLEEP_SECONDS = 15
FETCHED_AT = datetime.now(timezone.utc).isoformat()
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

RAW_OUTPUT = RAW_DIR / f"x_kol_posts_raw_{RUN_TS}.csv"
FILTERED_OUTPUT = FILTERED_DIR / f"x_kol_posts_market_filtered_{RUN_TS}.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw output: {RAW_OUTPUT}")
print(f"Filtered output: {FILTERED_OUTPUT}")

Project root: /home/mandesko/quant_project
Raw output: /home/mandesko/quant_project/Dataset/news/X/raw/x_kol_posts_raw_20260520_232036.csv
Filtered output: /home/mandesko/quant_project/Dataset/news/X/filtered/x_kol_posts_market_filtered_20260520_232036.csv


In [2]:
# Replace this starter list with your KOL handles.
# Keep handles without the leading '@'.
kols = [
    {"handle": "zerohedge", "name": "ZeroHedge"},
    {"handle": "elonmusk", "name": "Elon Musk"},
    {"handle": "DeItaone", "name": "Walter Bloomberg"},
    {"handle": "realDonaldTrump", "name": "Donald j. Trump"},
    {"handle": "KobeissiLetter", "name": "The Kobeissi Letter"},
    {"handle": "MrMBrown", "name": "Michael Brown"},
    {"handle": "BrianFeroldi", "name": "Brian Feroldi"},
    {"handle": "10kdiver", "name": "10k Diver"},
    {"handle": "CathieDWood", "name": "Cathie Wood"},
    {"handle": "howardlindzon", "name": "Howard Lindzon"},
    {"handle": "FluentInFinance", "name": "Andrew Lokenauth | TheFinanceNewsletter.com"}
]

kols = [
    {**kol, "handle": kol["handle"].lstrip("@").strip()}
    for kol in kols
    if kol.get("handle")
]

pd.DataFrame(kols)

,handle,name
0,zerohedge,ZeroHedge
1,elonmusk,Elon Musk
2,DeItaone,Walter Bloomberg
3,realDonaldTrump,Donald j. Trump
4,KobeissiLetter,The Kobeissi Letter
5,MrMBrown,Michael Brown
6,BrianFeroldi,Brian Feroldi
7,10kdiver,10k Diver
8,CathieDWood,Cathie Wood
9,howardlindzon,Howard Lindzon


In [6]:
import sys

sys.path.insert(0, str(DATA_DIR))
from twikit_patch import apply_twikit_user_patch

apply_twikit_user_patch()

client = Client("en-US")


def _is_cloudflare_block(exc: Exception) -> bool:
    message = str(exc).lower()
    return (
        "cloudflare" in message
        or "sorry, you have been blocked" in message
        or "attention required" in message
    )


async def login_with_cookie_reuse(client: Client, cookie_file: Path = COOKIE_FILE) -> Client:
    """Load reusable X cookies when available; otherwise log in once and save them."""
    from twikit.errors import Forbidden

    cookie_override = os.getenv("X_COOKIE_FILE", "").strip()
    cookie_path = Path(cookie_override).expanduser() if cookie_override else cookie_file

    if cookie_path.exists():
        client.load_cookies(str(cookie_path))
        print(f"Loaded cookies from {cookie_path}")
        return client

    username = os.getenv("X_USERNAME") or input("X username or phone: ").strip()
    email = os.getenv("X_EMAIL") or input("X email: ").strip()
    password = os.getenv("X_PASSWORD") or getpass("X password: ")

    try:
        await client.login(
            auth_info_1=username,
            auth_info_2=email,
            password=password,
        )
    except Forbidden as exc:
        if _is_cloudflare_block(exc):
            raise RuntimeError(
                "X login is blocked by Cloudflare (HTTP 403) on this network/IP. "
                "Open x.com in a normal browser on the same network and complete any challenge, "
                "or switch network and retry. If you already have a valid twikit cookies file, "
                "place it at Data/X/.x_cookies.json or set X_COOKIE_FILE to that path."
            ) from exc
        raise

    cookie_path.parent.mkdir(parents=True, exist_ok=True)
    client.save_cookies(str(cookie_path))
    print(f"Saved cookies to {cookie_path}")
    return client


client = await login_with_cookie_reuse(client)

# Important: keep Data/X/.x_cookies.json local and do not commit it.

Loaded cookies from /home/mandesko/quant_project/Data/X/.x_cookies.json


In [7]:
RAW_COLUMNS = [
    "handle", "kol_name", "user_id", "post_id", "created_at", "text", "hashtags", "urls",
    "url"
]


def safe_attr(obj, name, default=None):
    return getattr(obj, name, default)


def tweet_url(handle: str, tweet_id: str | None) -> str | None:
    if not tweet_id:
        return None
    return f"https://x.com/{handle}/status/{tweet_id}"


def normalize_tweet(kol: dict, user, tweet) -> dict:
    handle = kol["handle"]
    created_dt = safe_attr(tweet, "created_at_datetime")
    created_at = created_dt.isoformat() if created_dt is not None else safe_attr(tweet, "created_at")
    urls = safe_attr(tweet, "urls", None) or []
    hashtags = safe_attr(tweet, "hashtags", None) or []

    return {
        "handle": handle,
        "kol_name": kol.get("name"),
        "user_id": safe_attr(user, "id"),
        "post_id": safe_attr(tweet, "id"),
        "created_at": created_at,
        "text": safe_attr(tweet, "full_text", None) or safe_attr(tweet, "text", None),
        "hashtags": json.dumps(hashtags, ensure_ascii=False),
        "urls": json.dumps(urls, ensure_ascii=False, default=str),
        "url": tweet_url(handle, safe_attr(tweet, "id")),
    }


async def fetch_kol_timeline(client: Client, kol: dict, count: int) -> list[dict]:
    handle = kol["handle"]
    user = await client.get_user_by_screen_name(handle)
    tweets = await client.get_user_tweets(user.id, "Tweets", count=count)
    rows = [normalize_tweet(kol, user, tweet) for tweet in tweets]
    print(f"{handle}: fetched {len(rows)} posts")
    return rows


async def fetch_all_kol_timelines(kols: list[dict], count: int = POSTS_PER_KOL) -> pd.DataFrame:
    rows: list[dict] = []
    for idx, kol in enumerate(kols, start=1):
        try:
            rows.extend(await fetch_kol_timeline(client, kol, count=count))
        except Exception as exc:
            print(f"Error fetching @{kol['handle']}: {exc}")

        if idx < len(kols):
            delay = random.uniform(MIN_SLEEP_SECONDS, MAX_SLEEP_SECONDS)
            print(f"Sleeping {delay:.1f}s before next KOL")
            await asyncio.sleep(delay)

    return pd.DataFrame(rows, columns=RAW_COLUMNS)


df_raw = await fetch_all_kol_timelines(kols)
df_raw.to_csv(RAW_OUTPUT, index=False)
print(f"Wrote {len(df_raw)} raw rows to {RAW_OUTPUT}")
df_raw.head()

zerohedge: fetched 20 posts
Sleeping 5.3s before next KOL
elonmusk: fetched 20 posts
Sleeping 7.2s before next KOL
DeItaone: fetched 20 posts
Sleeping 9.1s before next KOL
realDonaldTrump: fetched 20 posts
Sleeping 8.1s before next KOL
KobeissiLetter: fetched 19 posts
Sleeping 12.3s before next KOL
MrMBrown: fetched 20 posts
Sleeping 15.0s before next KOL
BrianFeroldi: fetched 16 posts
Sleeping 9.4s before next KOL
10kdiver: fetched 2 posts
Sleeping 14.9s before next KOL
CathieDWood: fetched 20 posts
Sleeping 11.3s before next KOL
howardlindzon: fetched 19 posts
Sleeping 11.2s before next KOL
FluentInFinance: fetched 20 posts
Wrote 196 raw rows to /home/mandesko/quant_project/Dataset/news/X/raw/x_kol_posts_raw_20260520_232036.csv


,handle,kol_name,user_id,post_id,created_at,text,hashtags,urls,url
0,zerohedge,ZeroHedge,18856867,2057121476751364351,2026-05-20T15:29:14+00:00,taps the sign,[],[],https://x.com/zerohedge/status/205712147675136...
1,zerohedge,ZeroHedge,18856867,2057121447080874297,2026-05-20T15:29:07+00:00,"The AI Economy, Part 1: Looking Beyond The Fac...",[],"[{""display_url"": ""zerohedge.com/markets/ai-eco...",https://x.com/zerohedge/status/205712144708087...
2,zerohedge,ZeroHedge,18856867,2057121293871337518,2026-05-20T15:28:30+00:00,Maybe not this time: entire gap almost fully r...,[],[],https://x.com/zerohedge/status/205712129387133...
3,zerohedge,ZeroHedge,18856867,2057119936552517772,2026-05-20T15:23:06+00:00,Crude Extends Decline As Trump Says In 'Final ...,[],"[{""display_url"": ""zerohedge.com/geopolitical/i...",https://x.com/zerohedge/status/205711993655251...
4,zerohedge,ZeroHedge,18856867,2057119838179369297,2026-05-20T15:22:43+00:00,RT @JavierBlas: Say whatever you want about US...,[],[],https://x.com/zerohedge/status/205711983817936...


In [8]:
# Keyword buckets tuned for US large-cap / S&P 500–relevant social posts.
# Dropped bare tokens like "market", "stocks", "ai", "war" that match off-topic text.
keyword_groups = {
    "us_equities_index": [
        "s&p 500",
        "s&p500",
        "spx",
        "spy",
        "qqq",
        "iwm",
        "dia",
        "large cap",
        "large-cap",
        "megacap",
        "mega-cap",
        "mag 7",
        "mag seven",
        "magnificent seven",
        "blue chip",
        "wall street",
        "stock market",
        "equities",
        "nasdaq 100",
        "nasdaq composite",
        "russell 2000",
        "s&p",
    ],
    "macro_policy": [
        "fed",
        "fomc",
        "powell",
        "cpi",
        "core cpi",
        "ppi",
        "pce",
        "core pce",
        "inflation",
        "nfp",
        "payroll",
        "jobs report",
        "jolts",
        "unemployment",
        "gdp",
        "ism",
        "recession",
        "soft landing",
        "rate cut",
        "rate hike",
        "fed funds",
        "dot plot",
        "qt",
        "qe",
        "treasury",
        "10-year",
        "10 year",
        "yield curve",
        "yield",
        "real yield",
        "dollar index",
        "dxy",
    ],
    "earnings_corporate": [
        "earnings",
        "eps",
        "guidance",
        "raised guidance",
        "lowered guidance",
        "revenue",
        "profit warning",
        "earnings miss",
        "earnings beat",
        "eps beat",
        "eps miss",
        "buyback",
        "dividend",
        "dividend cut",
        "analyst upgrade",
        "analyst downgrade",
        "earnings call",
        "conference call",
        "same-store sales",
        "merger",
        "acquisition",
        "m&a",
        "takeover",
        "spin-off",
        "spinoff",
        "restructuring",
        "layoff",
        "job cuts",
        "shareholder",
    ],
    "market_risk": [
        "vix",
        "volatility",
        "breadth",
        "advance-decline",
        "advance decline",
        "selloff",
        "sell-off",
        "rally",
        "correction",
        "bear market",
        "bull market",
        "drawdown",
        "risk-on",
        "risk off",
        "sector rotation",
        "rotation",
        "flight to quality",
    ],
    "sectors_gics_etfs": [
        "xlk",
        "xlf",
        "xle",
        "xlv",
        "xli",
        "xly",
        "xlp",
        "xlu",
        "xlre",
        "xlb",
        "xlc",
        "vanguard",
        "spdr",
        "sector etf",
        "gics",
        "financials",
        "health care",
        "healthcare sector",
        "energy sector",
        "technology sector",
        "consumer staples",
        "consumer discretionary",
        "industrials sector",
        "materials sector",
        "utilities sector",
        "real estate sector",
        "communication services",
    ],
    "trade_geopolitics_commodities": [
        "trade war",
        "tariff",
        "sanction",
        "supply chain",
        "china trade",
        "taiwan",
        "russia",
        "ukraine",
        "middle east",
        "opec",
        "oil",
        "brent",
        "wti",
        "natural gas",
        "commodity",
    ],
    "semis_tech_infra": [
        "semiconductor",
        "semis",
        "chip",
        "chips",
        "foundry",
        "nvidia",
        "gpu",
        "data center",
        "datacenter",
        "hyperscaler",
        "cloud infrastructure",
        "cybersecurity",
        "generative ai",
        "artificial intelligence",
    ],
    "regulatory": [
        "sec",
        "ftc",
        "doj",
        "antitrust",
        "investigation",
        "subpoena",
        "settlement",
    ],
}

flat_keywords = sorted({kw for values in keyword_groups.values() for kw in values}, key=len, reverse=True)
cashtag_pattern = re.compile(r"(?<![A-Za-z0-9_])\$[A-Z]{1,6}(?:\.[A-Z]{1,2})?\b")


def matched_keywords(text: str | None) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    normalized = text.lower()
    return [kw for kw in flat_keywords if re.search(rf"(?<!\w){re.escape(kw.lower())}(?!\w)", normalized)]


def matched_event_buckets(keywords: list[str]) -> list[str]:
    return [
        bucket
        for bucket, bucket_keywords in keyword_groups.items()
        if any(keyword in bucket_keywords for keyword in keywords)
    ]


def extract_cashtags(text: str | None) -> list[str]:
    if not isinstance(text, str):
        return []
    return cashtag_pattern.findall(text.upper())


if df_raw.empty:
    df_filtered = df_raw.copy()
    for column in ["matched_keywords", "event_bucket", "cashtags", "contains_cashtag"]:
        df_filtered[column] = pd.Series(dtype="object")
else:
    df_filtered = df_raw.copy()
    df_filtered["matched_keywords_list"] = df_filtered["text"].apply(matched_keywords)
    df_filtered["event_bucket_list"] = df_filtered["matched_keywords_list"].apply(matched_event_buckets)
    df_filtered["cashtags_list"] = df_filtered["text"].apply(extract_cashtags)
    df_filtered["contains_cashtag"] = df_filtered["cashtags_list"].apply(bool)

    df_filtered = df_filtered[
        df_filtered["matched_keywords_list"].apply(bool) | df_filtered["contains_cashtag"]
    ].copy()

    df_filtered["matched_keywords"] = df_filtered["matched_keywords_list"].apply(lambda x: ",".join(x))
    df_filtered["event_bucket"] = df_filtered["event_bucket_list"].apply(lambda x: ",".join(x))
    df_filtered["cashtags"] = df_filtered["cashtags_list"].apply(lambda x: ",".join(x))
    df_filtered = df_filtered.drop(columns=["matched_keywords_list", "event_bucket_list", "cashtags_list"])

# Optional date window example:
# start_date = "2026-01-01"
# end_date = "2026-12-31"
# df_filtered = df_filtered[
#     pd.to_datetime(df_filtered["created_at"], errors="coerce").between(start_date, end_date)
# ]

df_filtered.to_csv(FILTERED_OUTPUT, index=False)
print(f"Wrote {len(df_filtered)} filtered rows to {FILTERED_OUTPUT}")
df_filtered.head()

Wrote 50 filtered rows to /home/mandesko/quant_project/Dataset/news/X/filtered/x_kol_posts_market_filtered_20260520_232036.csv


,handle,kol_name,user_id,post_id,created_at,text,hashtags,urls,url,contains_cashtag,matched_keywords,event_bucket,cashtags
4,zerohedge,ZeroHedge,18856867,2057119838179369297,2026-05-20T15:22:43+00:00,RT @JavierBlas: Say whatever you want about US...,[],[],https://x.com/zerohedge/status/205711983817936...,False,oil,trade_geopolitics_commodities,
5,zerohedge,ZeroHedge,18856867,2057119444187398541,2026-05-20T15:21:09+00:00,here we go again\n\n*TRUMP SAYS US IN 'FINAL S...,[],[],https://x.com/zerohedge/status/205711944418739...,False,"10-year,yield",macro_policy,
10,zerohedge,ZeroHedge,18856867,2057110625529020811,2026-05-20T14:46:07+00:00,Oil Prices Extend Decline After The Largest Cr...,[],"[{""display_url"": ""zerohedge.com/energy/oil-pri...",https://x.com/zerohedge/status/205711062552902...,False,"drawdown,oil","market_risk,trade_geopolitics_commodities",
14,zerohedge,ZeroHedge,18856867,2057108864605069727,2026-05-20T14:39:07+00:00,Oil Prices Extend Decline After Huge Inventory...,[],"[{""display_url"": ""zerohedge.com/energy/oil-pri...",https://x.com/zerohedge/status/205710886460506...,False,oil,trade_geopolitics_commodities,
40,DeItaone,Walter Bloomberg,2704294333,2057121075796865235,2026-05-20T15:27:38+00:00,YIELD ON 10-YR U.S. TREASURY NOTE LAST DOWN 8....,[],[],https://x.com/DeItaone/status/2057121075796865235,False,"treasury,yield",macro_policy,


In [9]:
# Small validation run before scaling.
# Set RUN_LIVE_TEST = True after authentication succeeds to fetch only the first 1-2 KOLs.
RUN_LIVE_TEST = False
TEST_POSTS_PER_KOL = 10
TEST_KOLS = kols[:2]

if RUN_LIVE_TEST:
    df_test = await fetch_all_kol_timelines(TEST_KOLS, count=TEST_POSTS_PER_KOL)
else:
    df_test = df_raw.head(min(len(df_raw), TEST_POSTS_PER_KOL * max(len(TEST_KOLS), 1))).copy()

missing_columns = sorted(set(RAW_COLUMNS) - set(df_test.columns))
print(f"Validation rows: {len(df_test)}")
print(f"Missing expected columns: {missing_columns or 'none'}")

if not df_test.empty:
    display(df_test[RAW_COLUMNS].head())

Validation rows: 20
Missing expected columns: none


,handle,kol_name,user_id,post_id,created_at,text,hashtags,urls,url
0,zerohedge,ZeroHedge,18856867,2057121476751364351,2026-05-20T15:29:14+00:00,taps the sign,[],[],https://x.com/zerohedge/status/205712147675136...
1,zerohedge,ZeroHedge,18856867,2057121447080874297,2026-05-20T15:29:07+00:00,"The AI Economy, Part 1: Looking Beyond The Fac...",[],"[{""display_url"": ""zerohedge.com/markets/ai-eco...",https://x.com/zerohedge/status/205712144708087...
2,zerohedge,ZeroHedge,18856867,2057121293871337518,2026-05-20T15:28:30+00:00,Maybe not this time: entire gap almost fully r...,[],[],https://x.com/zerohedge/status/205712129387133...
3,zerohedge,ZeroHedge,18856867,2057119936552517772,2026-05-20T15:23:06+00:00,Crude Extends Decline As Trump Says In 'Final ...,[],"[{""display_url"": ""zerohedge.com/geopolitical/i...",https://x.com/zerohedge/status/205711993655251...
4,zerohedge,ZeroHedge,18856867,2057119838179369297,2026-05-20T15:22:43+00:00,RT @JavierBlas: Say whatever you want about US...,[],[],https://x.com/zerohedge/status/205711983817936...
